# Pandas on meteorological data from the Max Planck Institute for Biogeochemistry

Credit: Some of the code in this tutorial is adapted from the excellent [TensorFlow tutorial on time series](https://www.tensorflow.org/tutorials/structured_data/time_series), which is licensed under the Apache-2.0 license. This is an excellent resource that you can consult if you want to study how to process time series with neural networks.

## Imports

In [ ]:
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import tensorflow as tf

## `pandas` import

Import `pandas` in the standard way.

In [ ]:
# Your code here

### Solution

In [ ]:
import pandas as pd

## Downloading the dataset

In [ ]:
zip_path = tf.keras.utils.get_file(
  origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
  fname="jena_climate_2009_2016.csv.zip",
  extract=True,
)
csv_path, _ = os.path.splitext(zip_path)
csv_path += "/jena_climate_2009_2016.csv"

## Reading data from the CSV file

Use `pandas.read_csv` to read the input file (stored at `csv_path`) in the `DataFrame` `df`.

You will need to make sure that the dates are parsed, and that the index column is `Date Time`.

In [ ]:
# Your code here

### Solution

In [ ]:
df = pd.read_csv(
  csv_path, parse_dates=True, date_format="%d.%m.%Y %H:%M:%S", index_col="Date Time"
)
df

## Analyzing `df`

Study `df` with the means seen in class. Determine:

- The variables in the data set (the columns)
- The number of samples
- The time between two samples
- The characteristics of the numerical variables

In [ ]:
# Your code here

### Solution

In [ ]:
print(f"DataFrame shape : {df.shape}")
print(f"DataFrame columns: {', '.join(df.columns)}")
print(f"Time between two successive sample: {df.index[1] - df.index[0]}")
df.describe().transpose()

## Outliers correction

As can be seen above, two columns have wind speed values at `-9999.0`, which is of course outliers.

Correct these values by replacing them with `0`.

Then check with a call to `describe` that the correction has been applied.

In [ ]:
# Your code here

### Solution

In [ ]:
wv = df["wv (m/s)"]
mask = df["wv (m/s)"] == -9999.0
wv[mask] = 0.0

max_wv = df["max. wv (m/s)"]
mask = max_wv == -9999.0
max_wv[mask] = 0.0

## Resampling

As we saw in the previous question, the time interval between two samples is very small at this point.

Resample the dataset using the function [`pandas.DataFrame.resample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) so that the measurements are averaged per day.

In [ ]:
# Your code here

### Solution

In [ ]:
df_resampled = df.resample("1D").mean()
print(f"DataFrame shape: {df_resampled.shape}")
print(
  f"Elapsed time between two samples: {df_resampled.index[1] - df_resampled.index[0]}"
)

## Comparisons

Calculate the average temperature and pressure (`T (degC)` and `p (mbar)`) for April 2014 and April 2015. Which was the warmer year? With the highest pressure?

In [ ]:
# Your code here

### Solution

In [ ]:
def compute_stats(date: str) -> pd.Series:
  return df_resampled.loc[date, ["T (degC)", "p (mbar)"]].mean()


stats_2014_04, stats_2015_04 = compute_stats("2014-04"), compute_stats("2015-04")
stats_2015_04 > stats_2014_04

## `DataFrame` visualization

Display the `T (degC)` & `p (mbar)` columns using the [`pandas.DataFrame.plot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html) method. You can use the argument `subplots=True` in your call to `plot` to make the result more readable.

The goal is to reproduce more succinctly the following result:

In [ ]:
_, (ax_t, ax_p) = plt.subplots(2, sharex=True)


def plot_column(ax: mpl.axes.Axes, column: str, color: str, xlabel: bool) -> None:
  x = df_resampled.index
  ax.plot(x, df_resampled[column], label=column, color=color)
  ax.set_xlim(x.min(), x.max())
  if xlabel:
    ax.set_xlabel(df_resampled.index.name)
  ax.legend()


plot_column(ax_t, "T (degC)", color="C0", xlabel=False)
plot_column(ax_p, "p (mbar)", color="C1", xlabel=True)
plt.show()

In [ ]:
# Your code here

### Solution

In [ ]:
df_resampled[["T (degC)", "p (mbar)"]].plot(subplots=True)
plt.show()

# For a specific month
df_resampled.loc["2014-01", ["T (degC)", "p (mbar)"]].plot(subplots=True)
plt.show()